In [3]:
import os
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

def fit_preprocessor(X_train: pd.DataFrame):
    """
    Fit the preprocessing pipeline on training data only.

    Drops constant/identifier columns:
    - 'EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'
    
    Encodes categorical columns appropriately.
    Scales numerical columns with StandardScaler.

    Returns fitted pipeline.
    """
    # Columns to drop (constant and ID)
    drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
    
    X_train = X_train.drop(columns=drop_cols, errors='ignore')

    # Separate numeric and categorical features
    numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

    # Debug
    # print("Numerical features:", numeric_features)
    # print("Categorical features:", categorical_features)

    # Preprocessing pipeline for numeric features
    numeric_transformer = StandardScaler()

    # Preprocessing pipeline for categorical features (OneHot)
    categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse=False)

    # Combine preprocessors
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'  # Drop other columns not specified
    )

    # Wrap in a Pipeline (can add feature engineering steps later)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

    # Fit pipeline on training data
    pipeline.fit(X_train)

    return pipeline

def transform_features(pipeline: Pipeline, X: pd.DataFrame):
    """
    Transform features using the fitted preprocessing pipeline.
    """
    drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
    X = X.drop(columns=drop_cols, errors='ignore')
    return pipeline.transform(X)

def save_preprocessor(pipeline: Pipeline, filepath: str):
    """
    Save the fitted preprocessing pipeline to the given filepath.
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    joblib.dump(pipeline, filepath)
    print(f"Preprocessing pipeline saved to {filepath}")

def load_preprocessor(filepath: str) -> Pipeline:
    """
    Load the preprocessing pipeline from the given filepath.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Pipeline file not found: {filepath}")
    pipeline = joblib.load(filepath)
    return pipeline
